In [1]:
import sys
import torch
sys.path.insert(0,"../src")

from cnn_project.train import train_model
from cnn_project.model import EuroSATModel
from cnn_project.evaluate import evaluate_model
from cnn_project.data import create_data_loaders
from pathlib import Path
import matplotlib.pyplot as plt
import optuna




/home/h171138/EuroSAT-Model/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("The model is using CUDA")
else:
    device = torch.device("cpu")
    print("The model is using CPU")

torch.manual_seed(42)
model = EuroSATModel().to(device)
train_loader, val_loader, test_loader= create_data_loaders(batch_size=64)

The model is using CUDA


In [3]:
import time
from torch.utils.data import DataLoader

images_to_decode = train_loader.dataset.dataset

for workers in [0, 2, 4, 8,10,10,12, 12 ,14,14,16 ,16,18,18]:
    loader = DataLoader(
        images_to_decode,
        batch_size=128,
        shuffle=False,
        num_workers=workers,
        persistent_workers=(workers > 0),
        prefetch_factor=2 if workers > 0 else None,
    )

    # First pass: start workers and warm the file cache.
    for images, labels in loader:
        pass

    # Second pass: measure images actually processed.
    count = 0
    start = time.perf_counter()

    for images, labels in loader:
        count += images.size(0)

    elapsed = time.perf_counter() - start
    print(
        f"{workers} workers: {count} images in {elapsed:.2f} s "
        f"({count / elapsed:.0f} images/s)"
    )

0 workers: 21600 images in 0.42 s (51999 images/s)
2 workers: 21600 images in 0.52 s (41375 images/s)
4 workers: 21600 images in 0.45 s (48409 images/s)
8 workers: 21600 images in 0.30 s (72814 images/s)
10 workers: 21600 images in 0.28 s (75830 images/s)
10 workers: 21600 images in 0.28 s (77455 images/s)
12 workers: 21600 images in 0.28 s (76891 images/s)
12 workers: 21600 images in 0.29 s (75425 images/s)
14 workers: 21600 images in 0.29 s (74500 images/s)
14 workers: 21600 images in 0.29 s (75383 images/s)
16 workers: 21600 images in 0.29 s (75475 images/s)
16 workers: 21600 images in 0.30 s (73160 images/s)


/home/h171138/EuroSAT-Model/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 18 worker processes in total. Our suggested max number of worker in current system is 16, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/home/h171138/EuroSAT-Model/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:437: UserWarning: This DataLoader will create 18 worker processes in total. Our suggested max number of worker in current system is 16, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_ration

18 workers: 21600 images in 0.29 s (74157 images/s)
18 workers: 21600 images in 0.28 s (78362 images/s)
